# v2 Model Review

Review the default v2 cross-sectional rank run, including signal quality, IC behavior, quantile spread returns, and strategy-versus-benchmark equity curves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.4f}'.format

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PRED_DIR = ROOT / 'outputs' / 'predictions'
METRICS_DIR = ROOT / 'outputs' / 'metrics'
RUN_PREFIX = 'ridge_regression_cross_sectional_rank'

predictions = pd.read_csv(PRED_DIR / f'{RUN_PREFIX}_predictions.csv', parse_dates=['date'])
portfolio_returns = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_portfolio_returns.csv', parse_dates=['date'])
metrics_summary = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_metrics_summary.csv')
backtest_summary = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_backtest_metrics.csv')
ic_by_date = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_ic_by_date.csv', parse_dates=['date'])
quantile_returns = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_quantile_returns.csv')
quantile_spread = pd.read_csv(METRICS_DIR / f'{RUN_PREFIX}_quantile_spread_returns.csv', parse_dates=['date'])

summary = pd.concat([metrics_summary, backtest_summary], axis=1)
summary.T.rename(columns={0: 'value'})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(ic_by_date['date'], ic_by_date['information_coefficient'], alpha=0.45, label='Daily IC')
ax.plot(ic_by_date['date'], ic_by_date['information_coefficient'].rolling(63).mean(), linewidth=2, label='63D Rolling Mean IC')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Information Coefficient Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Spearman IC')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(quantile_returns['quantile'].astype(str), quantile_returns['mean_realized_return'], color='#2f5d80')
ax.set_title('Mean 5D Forward Return by Prediction Quantile')
ax.set_xlabel('Prediction Quantile')
ax.set_ylabel('Mean Realized 5D Return')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(quantile_spread['date'], quantile_spread['spread_return'].cumsum(), linewidth=2, label='Top Minus Bottom Quantile')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Cumulative Quantile Spread Return')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Top Minus Bottom Quantile Return')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(portfolio_returns['date'], portfolio_returns['equity_curve'], linewidth=2, label='Strategy')
if 'benchmark_equity_curve' in portfolio_returns.columns:
    ax.plot(portfolio_returns['date'], portfolio_returns['benchmark_equity_curve'], linewidth=2, label='SPY Benchmark')
ax.set_title('Strategy vs Benchmark Equity Curve')
ax.set_xlabel('Date')
ax.set_ylabel('Equity Curve')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
sample = predictions.sample(min(len(predictions), 2500), random_state=42)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(sample['prediction'], sample['target'], alpha=0.3, s=10, color='#5b8c5a')
ax.set_title('Predicted Score vs Cross-Sectional Rank Target')
ax.set_xlabel('Predicted Score')
ax.set_ylabel('Cross-Sectional Rank Target')
plt.tight_layout()
plt.show()